# Vígil.ia — modelo base **só com o dataset de 12,5k**, otimizado pro Jetson

Um notebook, um modelo, uma fonte de dado: as **12.528 imagens do Roboflow**.
Sem fotos de celular, sem capturas do `vigil_deck`, sem vídeo. Nada de FT1/FT2/
FT3/FT4.

## Por que isso faz sentido agora

Domain shift é o gargalo estrutural do projeto — reapareceu nas três eras. A
resposta que sempre se tentou foi *adaptar o modelo ao domínio novo* (fine-tune
atrás de fine-tune). Aqui a estratégia se inverte: **adaptar o domínio ao
modelo**. A câmara de inspeção é construída para reproduzir o setup do dataset
— fundo preto fosco, luz difusa de cima, grãos sem se tocar. Se os dois
domínios forem o mesmo, o modelo base transfere sem fine-tune.

Isso dá um modelo **limpo e auditável**: cada caixa vem de um dataset público,
com licença MIT, e nenhum rótulo duvidoso do meio do caminho. É a linha de base
contra a qual o dataset próprio vai ser medido depois.

## O ponto que não pode passar batido

> **"Só o dataset de 12,5k" NÃO é o mesmo que "só fotos de 1 grão".**

O dataset tem uma imagem por grão. A câmara vai ver **~26 grãos por recorte**.
Um modelo treinado só em foto solta não aprende densidade, oclusão, nem o prior
de quantos objetos existem no quadro — e a família DETR depende justamente
desse prior.

A solução é **renderizar cenas multi-grão a partir dos recortes do próprio
12,5k**. Continua sendo a mesma fonte de dado; muda só a apresentação. E,
como agora a geometria da câmara é conhecida, as cenas são geradas **na escala
do rig**: grão em ~89 px, fundo com a queda de luz do ring light, densidade
igual à da esteira.

## O que este notebook faz de diferente do `treino_rfdetr_small_completo.ipynb`

| | notebook antigo | este |
|---|---|---|
| Fontes de dado | 12,5k + fotos reais + vídeo + capturas | **só 12,5k** |
| Estágios | base → FT1 → FT2 → FT3 → FT4 | **um treino só** |
| Tamanho do grão nas cenas | faixa genérica 60-150 px | **derivado da óptica do rig** |
| Fundo das cenas | cinza uniforme 20-130 | **fundo da câmara: preto fosco + vinheta do ring light** |
| Proporção de defeito | uniforme entre classes | **varia por cena** (lote bom e lote ruim) |
| Métrica | mAP | mAP + **recall por classe + curva premium × não-premium** |

Tempo estimado (A100): **~30-40 min** de dataset + **~2h30-3h30** de treino.

## 0. Setup

In [ ]:
# >=1.9: RFDETRLarge = 704px nativo (a deprecada de 560px virou
# RFDETRLargeDeprecated) e load_pretrain_weights interpola o PE.
!pip -q install "rfdetr[train,loggers]>=1.9"
from importlib.metadata import version
print('rfdetr', version('rfdetr'))

## 1. Config

Dois blocos: o que define o **modelo** e o que define o **rig**. O segundo é o
que faz as cenas sintéticas saírem na escala certa — se a calibração com a
régua (`jetson/calibrar_rig.py`) devolver outro px/mm, **é aqui que se
corrige**, e o dataset inteiro acompanha.

In [ ]:
import os, glob, shutil
import torch
from google.colab import drive
drive.mount('/content/drive')

# ============================ MODELO ============================
# As variantes têm praticamente o MESMO tamanho (~30-34M params); o que muda é
# a RESOLUÇÃO NATIVA. A 'large' é na prática "a small a 704px".
#   nano 384 | small 512 | medium 576 | large 704   (todas Apache 2.0)
# Nunca passar `resolution=` custom: o checkpoint pré-treinado de cada variante
# foi treinado NA resolução dela, então o positional embedding casa exato.
VARIANTE = 'large'
RES = {'nano': 384, 'small': 512, 'medium': 576, 'large': 704}[VARIANTE]
SIZE = RES                      # canvas do dataset = entrada do modelo

# ============================== RIG ==============================
# Vem de jetson/PADRAO_CAPTURA.md §1c e jetson/calcular_vazao.py.
# CONFIRA com a régua antes de rodar: `python3 jetson/calibrar_rig.py`.
PX_POR_MM   = 12.7              # câmera a ~9,3 cm, sensor cheio 3280x2464
GRAO_MM     = 7.0               # soja: 6-8 mm
ESPACO_MM   = GRAO_MM * 1.4     # centro a centro na monocamada

# Escala do grão nas cenas: centrada no que o rig entrega, com folga para o
# tamanho real do grão (6-8 mm) e para erro de calibração da distância.
GRAO_PX     = GRAO_MM * PX_POR_MM                    # ~89 px
GRAO_PX_MIN = int(GRAO_PX * 0.75)                    # ~67
GRAO_PX_MAX = int(GRAO_PX * 1.35)                    # ~120

# Densidade: quantos grãos cabem no recorte, em monocamada cheia.
_lado_mm  = SIZE / PX_POR_MM                         # ~55 mm
_max_grao = int((_lado_mm / ESPACO_MM) ** 2)         # ~32
GRAOS_MIN, GRAOS_MAX = max(3, int(_max_grao * 0.4)), _max_grao

# Fundo da câmara: cartolina preta fosca + ring light (centro mais claro que a
# borda). Bem mais escuro e mais estreito que a faixa genérica de antes.
FUNDO_MIN, FUNDO_MAX = 8, 45
VINHETA_MAX = 30                # quanto o centro fica acima da borda

# Prior de defeito. Um lote real é majoritariamente intacto, MAS treinar tudo
# com a mesma proporção ensina o prior em vez da contagem — o modelo passa a
# resistir a chamar um lote ruim de ruim. Por isso cada cena sorteia a própria
# taxa de defeito nesta faixa: o modelo vê lote bom E lote ruim.
DEFEITO_MIN, DEFEITO_MAX = 0.02, 0.65

# ============================ DATASET ============================
N_CENAS_TRAIN = 4000
N_CENAS_VAL   = 400
BLUR_FRAC     = 0.40            # fração das cenas com motion blur (esteira)
# Fotos SOLTAS entram só como tempero: o rig nunca vê 1 grão sozinho ocupando o
# quadro, e encher o treino disso ensina uma tarefa que não existe em operação.
FRAC_SOLTAS   = 0.25

# ============================= TREINO ============================
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9
GPU_NOME = torch.cuda.get_device_name(0)
# rfdetr pede batch TOTAL 16 (batch_size x grad_accum_steps). A memória de
# ativação cresce com RES^2: 704 custa ~1,9x o de 512.
_GRANDE = RES >= 576
if VRAM_GB > 30:                      # A100 40GB+
    BATCH, GRAD_ACCUM = (8, 2) if _GRANDE else (16, 1)
elif VRAM_GB > 20:                    # L4 24GB, A10, RTX 4090…
    BATCH, GRAD_ACCUM = (2, 8) if _GRANDE else (4, 4)
else:                                 # T4 16GB
    BATCH, GRAD_ACCUM = (1, 16) if _GRANDE else (4, 4)

EPOCAS    = 40
LR        = 1e-4
PACIENCIA = 10

# ============================ CAMINHOS ===========================
CLS_BASE_CANDS = [
    '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder',
    '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)',
]
CLS_BASE = next((p for p in CLS_BASE_CANDS if os.path.isdir(p)), None)
assert CLS_BASE, 'dataset 12,5k não encontrado:\n  ' + '\n  '.join(CLS_BASE_CANDS)

# A resolução ENTRA no nome: o treino pula se o arquivo existir, e sem o sufixo
# uma rodada em resolução nova reaproveitaria em silêncio um .pth de outra.
BASE_PTH = f'/content/drive/MyDrive/soja_rfdetr_{VARIANTE}_{SIZE}_base12k.pth'
DET_DIR  = f'/content/soja_base12k_{SIZE}'
COCO_DIR = f'/content/coco_base12k_{SIZE}'

print(f'GPU     : {GPU_NOME} ({VRAM_GB:.0f} GB)')
print(f'modelo  : {VARIANTE} @ {RES}px | batch {BATCH} x accum {GRAD_ACCUM}')
print(f'dataset : {CLS_BASE}')
print(f'rig     : {PX_POR_MM} px/mm -> grão {GRAO_PX:.0f} px '
      f'(cenas de {GRAO_PX_MIN} a {GRAO_PX_MAX} px)')
print(f'          recorte de {SIZE}px = {_lado_mm:.0f} mm -> '
      f'{GRAOS_MIN}-{GRAOS_MAX} grãos por cena')
print(f'saída   : {BASE_PTH}')

## 2. Funções de dataset

As de segmentação e conversão vêm do pipeline já validado
(`treino_rfdetr_small_completo.ipynb`): pseudo-rótulo Otsu, recorte com máscara
erodida (tira o halo, a caixa cola no grão), COCO JSON para o `rfdetr`.

O que é novo é `make_scene`: fundo da câmara, escala do rig e proporção de
defeito variável por cena.

In [ ]:
import glob, json, unicodedata, cv2
import numpy as np
from collections import Counter, defaultdict
from PIL import Image

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
PREMIUM = 'intact'
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']       # pasta que NÃO é classe — sempre excluir
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}


def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()


def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None


def collect_base(base_dir):
    """Acha train/valid/test em qualquer profundidade dentro do 12,5k."""
    items = []
    for root, dirs, _ in os.walk(base_dir):
        for d in list(dirs):
            sp = SPLIT_MAP.get(d.lower())
            if sp is None:
                continue
            split_dir = os.path.join(root, d)
            for folder in sorted(os.listdir(split_dir)):
                cls = class_of(folder)
                if cls is None:
                    continue
                for p in glob.glob(os.path.join(split_dir, folder, '*')):
                    if p.lower().endswith(IMG_EXT):
                        items.append((p, cls, sp))
            dirs.remove(d)
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    print('por classe:', {NAMES[c]: n for c, n in
                          sorted(Counter(c for _, c, _ in items).items())})
    return items


def otsu_box(img):
    """Caixa do grão por limiarização — funciona porque o fundo é controlado."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)


def extract_cutout(img):
    """Recorte + máscara do grão, para colar nas cenas sintéticas."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    bx, by, bw, bh = cv2.boundingRect(c)
    if area < 0.55 * bw * bh:      # solidez: grão é compacto; blob esfarrapado sai
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    mask = cv2.erode(mask, np.ones((3, 3), np.uint8))   # tira o halo
    ys, xs = np.where(mask > 0)
    if not len(xs):
        return None
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    return img[y0:y1, x0:x1], mask[y0:y1, x0:x1]


def letterbox(img, size=None):
    size = size or SIZE
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top


def motion_blur(img, rng):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)


def fundo_camara(size, rng):
    """Cartolina preta fosca iluminada por ring light.

    O ring light é um anel em volta da lente: o centro do quadro recebe mais luz
    que a borda. Reproduzir essa vinheta importa porque o modelo não pode
    aprender "borda escura => sem grão" — na esteira entra e sai grão pela borda
    o tempo todo.
    """
    base = float(rng.integers(FUNDO_MIN, FUNDO_MAX))
    y, x = np.mgrid[0:size, 0:size].astype(np.float32)
    c = (size - 1) / 2
    r = np.hypot(x - c, y - c) / (c * np.sqrt(2))
    vinheta = float(rng.uniform(0, VINHETA_MAX)) * (1 - r ** 2)
    campo = base + vinheta + rng.normal(0, 4, (size, size))
    return np.clip(campo[..., None].repeat(3, 2), 0, 255).astype(np.uint8)


def sorteia_classes(pool_por_classe, n, rng):
    """Escolhe as classes dos n grãos de uma cena.

    Cada cena sorteia a PRÓPRIA taxa de defeito. Sem isso, o modelo aprende o
    prior ("quase tudo é intacto") em vez de contar o que vê, e passa a resistir
    a classificar um lote ruim como ruim.
    """
    taxa = float(rng.uniform(DEFEITO_MIN, DEFEITO_MAX))
    defeitos = [c for c in range(5) if NAMES[c] != PREMIUM and pool_por_classe.get(c)]
    idx_prem = NAMES.index(PREMIUM)
    escolhas = []
    for _ in range(n):
        if defeitos and rng.random() < taxa:
            escolhas.append(defeitos[int(rng.integers(len(defeitos)))])
        elif pool_por_classe.get(idx_prem):
            escolhas.append(idx_prem)
        elif defeitos:
            escolhas.append(defeitos[int(rng.integers(len(defeitos)))])
    return escolhas


def make_scene(pool, rng, size=None):
    """Cena multi-grão na ESCALA DO RIG, com fundo da câmara.

    `pool` é {classe: [(crop, mask), …]}. Devolve (canvas, [(cls, cx, cy, w, h)]),
    tudo normalizado.
    """
    size = size or SIZE
    canvas = fundo_camara(size, rng)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    n = int(rng.integers(GRAOS_MIN, GRAOS_MAX + 1))
    for cls in sorteia_classes(pool, n, rng):
        crop, mask = pool[cls][int(rng.integers(len(pool[cls])))]
        alvo = int(rng.integers(GRAO_PX_MIN, GRAO_PX_MAX + 1))
        s = alvo / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        colocado = False
        for _ in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            # até 15% de encosto: no rig os grãos vêm espalhados, mas encostam
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                colocado = True
                break
        if not colocado:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size,
                      gw / size, gh / size))
    return canvas, boxes


def yolo_to_coco(split_srcs, out_dir, names=NAMES):
    """O rfdetr treina com COCO JSON (`_annotations.coco.json` por split)."""
    shutil.rmtree(out_dir, ignore_errors=True)
    cats = [{'id': i + 1, 'name': n, 'supercategory': 'soja'} for i, n in enumerate(names)]
    for split, srcs in split_srcs.items():
        os.makedirs(f'{out_dir}/{split}', exist_ok=True)
        images, anns = [], []
        img_id = ann_id = 0
        for img_dir, lbl_dir in srcs:
            for p in sorted(glob.glob(os.path.join(img_dir, '*.jpg'))):
                stem = os.path.splitext(os.path.basename(p))[0]
                lp = os.path.join(lbl_dir, stem + '.txt')
                if not os.path.exists(lp):
                    continue
                with Image.open(p) as im:
                    w, h = im.size
                fn = f'{img_id:06d}_{os.path.basename(p)}'
                try:
                    os.link(p, f'{out_dir}/{split}/{fn}')
                except OSError:
                    shutil.copy(p, f'{out_dir}/{split}/{fn}')
                images.append({'id': img_id, 'file_name': fn, 'width': w, 'height': h})
                for line in open(lp):
                    parts = line.split()
                    if len(parts) != 5:
                        continue
                    c = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:])
                    anns.append({'id': ann_id, 'image_id': img_id, 'category_id': c + 1,
                                 'bbox': [round((cx - bw / 2) * w, 2),
                                          round((cy - bh / 2) * h, 2),
                                          round(bw * w, 2), round(bh * h, 2)],
                                 'area': round(bw * w * bh * h, 2), 'iscrowd': 0})
                    ann_id += 1
                img_id += 1
        json.dump({'images': images, 'annotations': anns, 'categories': cats},
                  open(f'{out_dir}/{split}/_annotations.coco.json', 'w'))
        print(f'{out_dir}/{split}: {len(images)} imgs, {len(anns)} caixas')
    return out_dir


print('funções prontas')

## 3. Construir o dataset

Duas passadas sobre o 12,5k:

1. **Recortes** — cada imagem vira um grão recortado com máscara, guardado num
   pool por classe e por split. É a matéria-prima das cenas.
2. **Fotos soltas** — uma fração entra como está (letterbox), só para o modelo
   não estranhar um grão isolado.

O split `train`/`valid` do próprio dataset é respeitado: **nenhum grão do val
aparece numa cena de treino**. Sem isso a validação mediria memorização.

In [ ]:
import time

def construir(items, out_dir, n_train, n_val, blur_frac, frac_soltas):
    shutil.rmtree(out_dir, ignore_errors=True)
    for sp in ('train', 'val'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)

    # o dataset tem split 'test' também; ele não é usado aqui (o juiz final é o
    # conjunto anotado no rig, que ainda não existe) — vira treino.
    pools = {'train': defaultdict(list), 'val': defaultdict(list)}
    rng_solta = np.random.default_rng(11)
    n_solta = {'train': 0, 'val': 0}
    t0, sem_recorte, sem_caixa = time.time(), 0, 0

    for i, (path, cls, sp) in enumerate(items):
        if i % 1000 == 0:
            print(f'  {i}/{len(items)}  ({time.time()-t0:.0f}s)', flush=True)
        sp = 'val' if sp == 'val' else 'train'
        img = cv2.imread(path)
        if img is None:
            continue

        cut = extract_cutout(img)
        if cut is None:
            sem_recorte += 1
        else:
            pools[sp][cls].append(cut)

        if rng_solta.random() >= frac_soltas:
            continue
        box = otsu_box(img)
        if box is None:
            sem_caixa += 1
            continue
        h0, w0 = img.shape[:2]
        lb, s, left, top = letterbox(img)
        cx, cy, ww, hh = box
        stem = f'solta_{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(
            f'{cls} {(cx*w0*s+left)/SIZE:.6f} {(cy*h0*s+top)/SIZE:.6f} '
            f'{(ww*w0*s)/SIZE:.6f} {(hh*h0*s)/SIZE:.6f}')
        n_solta[sp] += 1

    print(f'\nrecortes: ' + ' | '.join(
        f'{sp}: ' + str({NAMES[c]: len(v) for c, v in sorted(pools[sp].items())})
        for sp in ('train', 'val')))
    print(f'fotos soltas: {n_solta} | sem recorte: {sem_recorte} | sem caixa: {sem_caixa}')
    # Classe sem recorte é falha SILENCIOSA: o make_scene simplesmente não
    # sorteia ela, o modelo nunca aprende a prever, e o sintoma só aparece lá
    # no recall — depois de horas de GPU. Falha aqui, de graça.
    for sp in ('train', 'val'):
        vazias = [NAMES[c] for c in range(5) if not pools[sp].get(c)]
        assert not vazias, (
            f'sem nenhum recorte de {vazias} no split {sp}. O extract_cutout '
            'segmenta por SATURAÇÃO — confira se as imagens dessa classe têm '
            'cor (não estão em tons de cinza) e se o fundo está escuro.')

    for sp, n, seed in (('train', n_train, 42), ('val', n_val, 123)):
        rng = np.random.default_rng(seed)
        feitas = 0
        for j in range(n):
            if j % 500 == 0:
                print(f'  cenas {sp} {j}/{n}…', flush=True)
            canvas, boxes = make_scene(pools[sp], rng)
            if not boxes:
                continue
            if rng.random() < blur_frac:
                canvas = motion_blur(canvas, rng)
            stem = f'cena_{sp}_{j:05d}'
            cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', canvas,
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(
                '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'
                          for c, cx, cy, w, h in boxes))
            feitas += 1
        print(f'cenas {sp}: {feitas}')
    return pools


if os.path.isdir(f'{DET_DIR}/images/train'):
    print('dataset já construído:', DET_DIR)
    POOLS = None
else:
    POOLS = construir(collect_base(CLS_BASE), DET_DIR,
                      N_CENAS_TRAIN, N_CENAS_VAL, BLUR_FRAC, FRAC_SOLTAS)

if not os.path.isdir(f'{COCO_DIR}/train'):
    # o rfdetr espera os três splits; 'test' aponta pro mesmo val (não há
    # conjunto de teste honesto ainda — o juiz final é o rig, que não existe)
    yolo_to_coco({'train': [(f'{DET_DIR}/images/train', f'{DET_DIR}/labels/train')],
                  'valid': [(f'{DET_DIR}/images/val', f'{DET_DIR}/labels/val')],
                  'test':  [(f'{DET_DIR}/images/val', f'{DET_DIR}/labels/val')]},
                 COCO_DIR)
else:
    print('COCO já pronto:', COCO_DIR)

## 3b. Conferir com os olhos antes de gastar 3 h de GPU

Duas checagens que já pegaram erro real neste projeto: caixa deslocada
(letterbox aplicado na ordem errada) e distribuição de classe torta entre train
e val (o "melhor checkpoint" escolhido por uma régua errada).

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def estatisticas(split):
    labs = sorted(glob.glob(f'{DET_DIR}/labels/{split}/*.txt'))
    cls_cnt, lados, por_img = Counter(), [], []
    for p in labs:
        linhas = [l.split() for l in open(p).read().splitlines() if l.strip()]
        por_img.append(len(linhas))
        for c, cx, cy, w, h in linhas:
            cls_cnt[NAMES[int(c)]] += 1
            lados.append((float(w) + float(h)) / 2 * SIZE)
    tot = sum(cls_cnt.values())
    print(f'\n[{split}] {len(labs)} imagens, {tot} caixas')
    for n in NAMES:
        print(f'   {n:14s} {cls_cnt[n]:7d}  ({100*cls_cnt[n]/max(tot,1):5.1f}%)')
    print(f'   grão: {np.mean(lados):.0f} px em média '
          f'(p5 {np.percentile(lados,5):.0f}, p95 {np.percentile(lados,95):.0f}) '
          f'| alvo do rig: {GRAO_PX:.0f} px')
    print(f'   caixas por imagem: mediana {int(np.median(por_img))}, '
          f'máx {max(por_img)}')
    return cls_cnt

c_tr, c_va = estatisticas('train'), estatisticas('val')
# distribuição parecida entre os splits: senão o val vira uma régua torta
p_tr = np.array([c_tr[n] for n in NAMES], float); p_tr /= p_tr.sum()
p_va = np.array([c_va[n] for n in NAMES], float); p_va /= p_va.sum()
dif = np.abs(p_tr - p_va).max()
print(f'\nmaior diferença de proporção train x val: {100*dif:.1f} pp'
      + ('  OK' if dif < 0.08 else '  ATENÇÃO: val com distribuição diferente'))

# amostra com as caixas desenhadas
CORES = [(170,100,210),(60,200,200),(90,200,90),(255,160,60),(70,70,235)]
amostra = sorted(glob.glob(f'{DET_DIR}/images/train/cena_*.jpg'))[:4] + \
          sorted(glob.glob(f'{DET_DIR}/images/train/solta_*.jpg'))[:2]
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, p in zip(axes.ravel(), amostra):
    img = cv2.imread(p)
    for l in open(p.replace('/images/', '/labels/').replace('.jpg', '.txt')):
        c, cx, cy, w, h = l.split(); c = int(c)
        cx, cy, w, h = (float(v) * SIZE for v in (cx, cy, w, h))
        cv2.rectangle(img, (int(cx-w/2), int(cy-h/2)), (int(cx+w/2), int(cy+h/2)),
                      CORES[c], 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis('off')
    ax.set_title(os.path.basename(p), fontsize=8)
plt.tight_layout(); plt.show()
print('As caixas TÊM que colar no grão. Se estiverem deslocadas ou frouxas,')
print('pare aqui — treinar em cima disso desperdiça a GPU.')

## 4. Treino — um estágio só

Sem cadeia de fine-tunings: **COCO → 12,5k**, e acabou. A cadeia FT1→FT4 do
notebook antigo existia para arrancar sinal de pouquíssima foto real; aqui não
há foto real nenhuma, de propósito.

Salva no Drive e **pula se já existir** — se a sessão do Colab cair, rode de
novo que ele retoma.

In [ ]:
import rfdetr

Modelo = {'nano': rfdetr.RFDETRNano, 'small': rfdetr.RFDETRSmall,
          'medium': rfdetr.RFDETRMedium, 'large': rfdetr.RFDETRLarge}[VARIANTE]
print(f'{Modelo.__name__} @ {RES}px')

if os.path.exists(BASE_PTH):
    print('já treinado, no Drive:', BASE_PTH)
else:
    out = f'/content/runs/{VARIANTE}_{SIZE}_base12k'
    m = Modelo()                       # <- parte do COCO, sem resolution=
    m.train(dataset_dir=COCO_DIR, epochs=EPOCAS, lr=LR,
            batch_size=BATCH, grad_accum_steps=GRAD_ACCUM,
            early_stopping=True, early_stopping_patience=PACIENCIA,
            tensorboard=False, wandb=False, output_dir=out)
    melhor = os.path.join(out, 'checkpoint_best_total.pth')
    assert os.path.exists(melhor), f'checkpoint_best_total.pth não apareceu em {out}'
    shutil.copy(melhor, BASE_PTH)
    print('salvo:', BASE_PTH)

## 5. Avaliação — recall por classe e a decisão premium

mAP é a métrica do detector; **não é a métrica do produto**. O que decide se o
MVP presta é:

- **Recall por classe** — quanto de cada defeito o modelo encontra. `immature` e
  `spotted` foram historicamente as classes fracas.
- **Recall de defeito na binária premium × não-premium** — quantos grãos ruins
  passam como premium. É o erro caro.
- **Perda de rendimento** — quanto grão bom é descartado à toa.

> ⚠️ Este val é feito das **mesmas 12,5k imagens** do treino (splits
> diferentes). Ele mede o quanto o modelo aprendeu a tarefa, **não** o quanto
> transfere para a câmara real. Enquanto não existir o conjunto anotado no rig,
> nenhum número aqui pode ser apresentado como "acurácia da máquina".

In [ ]:
modelo = Modelo(pretrain_weights=BASE_PTH)
VAL_DIR = f'{COCO_DIR}/valid'
coco = json.load(open(f'{VAL_DIR}/_annotations.coco.json'))
gt_por_img = defaultdict(list)
for a in coco['annotations']:
    x, y, w, h = a['bbox']
    gt_por_img[a['image_id']].append((a['category_id'] - 1, x, y, x + w, y + h))
arquivos = {im['id']: im['file_name'] for im in coco['images']}

CONF = 0.35
IOU_ACERTO = 0.5
N_AVALIAR = 250      # subconjunto: o val inteiro é lento e não muda a conclusão


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0


# ---------- passada 1: só prever ----------
# Nada é comparado ainda. O off-by-one de classe já apareceu 3x neste projeto
# (COCO é 1-indexado, o rfdetr pode devolver id 0- ou 1-indexado), e deduzir a
# base OLHANDO UMA IMAGEM é frágil: uma imagem só de `intact` não mostra nem o
# id 0 nem o id 5, e a dedução sai errada em silêncio. Por isso: prever tudo,
# decidir a base uma vez, e só então casar.
ids = sorted(arquivos)[:N_AVALIAR]
predicoes = {}
todos_ids = []
for k, iid in enumerate(ids):
    if k % 50 == 0:
        print(f'  prevendo {k}/{len(ids)}…', flush=True)
    img = Image.open(os.path.join(VAL_DIR, arquivos[iid])).convert('RGB')
    det = modelo.predict(img, threshold=CONF)
    cls = np.asarray(det.class_id).reshape(-1) if det.class_id is not None else np.empty(0, int)
    caixas = np.asarray(det.xyxy).reshape(-1, 4)
    confs = (np.asarray(det.confidence).reshape(-1) if det.confidence is not None
             else np.ones(len(caixas)))
    predicoes[iid] = (cls, caixas, confs)
    todos_ids += cls.tolist()

assert todos_ids, ('nenhuma detecção acima de conf=%.2f no val inteiro. '
                   'Ou o modelo não treinou, ou o conf está alto demais.' % CONF)
OFFSET = 1 if min(todos_ids) >= 1 else 0
print(f'\nids de classe observados: {sorted(set(todos_ids))}')
print(f'offset detectado: {OFFSET} '
      f'({"1-indexado, como o COCO" if OFFSET else "0-indexado"})')
if max(todos_ids) - OFFSET >= len(NAMES):
    print('ATENÇÃO: id acima do nº de classes — confira o mapeamento antes de ler os números')

# ---------- passada 2: casar ----------
pares = []            # (classe_real, classe_predita, confiança)
nao_achados = Counter()   # grão que existe e o detector não achou
falso_fundo = Counter()   # detecção que não corresponde a grão nenhum
for iid in ids:
    cls, caixas, confs = predicoes[iid]
    usados = set()
    for gt_c, *gt_box in gt_por_img.get(iid, []):
        melhor, melhor_iou = None, IOU_ACERTO
        for j in range(len(caixas)):
            if j in usados:
                continue
            v = iou(gt_box, caixas[j])
            if v > melhor_iou:
                melhor, melhor_iou = j, v
        if melhor is None:
            nao_achados[NAMES[gt_c]] += 1
            continue
        usados.add(melhor)
        pares.append((gt_c, int(cls[melhor]) - OFFSET, float(confs[melhor])))
    for j in range(len(caixas)):
        if j not in usados:
            c = int(cls[j]) - OFFSET
            falso_fundo[NAMES[c] if 0 <= c < len(NAMES) else f'id{c}'] += 1

print(f'\n{len(pares)} grãos casados')
print(f'não encontrados (o detector perdeu): {dict(nao_achados) or "nenhum"}')
print(f'falso positivo de fundo (caixa sem grão): {dict(falso_fundo) or "nenhum"}')

In [ ]:
# ---------- matriz de confusão + recall por classe ----------
M = np.zeros((5, 5), int)
fora = 0
for gt, pr, _ in pares:
    if 0 <= pr < 5:
        M[gt, pr] += 1
    else:
        fora += 1
if fora:
    print(f'ATENÇÃO: {fora} predições com id fora de 0..4 — mapeamento suspeito\n')

print('matriz de confusão (linha = verdade, coluna = predito)')
print('             ' + ''.join(f'{n[:8]:>10s}' for n in NAMES))
for i, n in enumerate(NAMES):
    print(f'{n:12s} ' + ''.join(f'{M[i,j]:10d}' for j in range(5)))

print(f'\n{"classe":14s} {"grãos":>7s} {"recall":>8s} {"precisão":>9s}'
      f' {"perdidos":>9s} {"fp fundo":>9s}')
for i, n in enumerate(NAMES):
    # recall CONTA o que o detector nem achou — senão o número fica bonito de
    # mentira: um modelo que só detecta o que é fácil teria recall alto.
    suporte = M[i].sum() + nao_achados[n]
    preditos = M[:, i].sum() + falso_fundo[n]
    rec = M[i, i] / suporte if suporte else 0.0
    prec = M[i, i] / preditos if preditos else 0.0
    print(f'{n:14s} {suporte:7d} {rec:8.3f} {prec:9.3f} '
          f'{nao_achados[n]:9d} {falso_fundo[n]:9d}')

tot = sum(M[i].sum() + nao_achados[n] for i, n in enumerate(NAMES))
print(f'\nacerto global: {M.trace()}/{tot} = {M.trace()/max(tot,1):.3f}')
print('Compare com o histórico: immature (recall 0,04) e spotted (0,30) eram as')
print('classes fracas. Se aqui elas subirem, o ganho veio de rótulo consistente.')

In [ ]:
# ---------- a métrica do produto: premium x não-premium ----------
# Aqui não há votação por rastreamento (isso é do app, com o grão em movimento);
# o que se mede é a decisão por observação única, que é o piso do sistema.
IDX_PREM = NAMES.index(PREMIUM)

def curva(limiar):
    """Grão vira não-premium se a confiança do defeito predito passar o limiar."""
    vp = fp = vn = fn = 0
    for gt, pr, cf in pares:
        real_def = (gt != IDX_PREM)
        pred_def = (0 <= pr < 5 and pr != IDX_PREM and cf >= limiar)
        vp += real_def and pred_def
        fn += real_def and not pred_def
        fp += (not real_def) and pred_def
        vn += (not real_def) and not pred_def
    return vp, fp, vn, fn

print(f'{"limiar":>7s} {"recall defeito":>15s} {"precisão":>9s} '
      f'{"perda rendim.":>14s}   (defeito que passa / grão bom descartado)')
for t in (0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.90):
    vp, fp, vn, fn = curva(t)
    rec = vp / max(vp + fn, 1)
    prec = vp / max(vp + fp, 1)
    perda = fp / max(fp + vn, 1)
    print(f'{t:7.2f} {rec:15.3f} {prec:9.3f} {perda:14.3f}'
          f'   {fn:5d} passam / {fp:5d} descartados')

print('\nComo escolher o ponto: subir o limiar deixa passar mais defeito e')
print('descarta menos grão bom. Qual dos dois erros é mais caro é decisão de')
print('negócio, não de engenharia — e é daqui que saem os RATIOS do app,')
print('hoje valores escolhidos na mão.')

## 6. Export ONNX → Jetson

O `.engine` do TensorRT é **atado ao aparelho e à versão do TensorRT** — não dá
para gerar aqui. O que sai do Colab é o ONNX; o engine nasce no Jetson.

In [ ]:
!pip -q install onnx onnxruntime
import onnx as _onnx

out = f'/content/export_{VARIANTE}_{SIZE}'
os.makedirs(out, exist_ok=True)
m = Modelo(pretrain_weights=BASE_PTH)
m.export(output_dir=out)          # sem resolution=: herda a nativa da variante
onnx_path = next((p for p in [f'{out}/inference_model.onnx', *glob.glob(f'{out}/*.onnx')]
                  if os.path.exists(p)), None)
assert onnx_path, f'export não gerou .onnx em {out}'

g = _onnx.load(onnx_path)
_onnx.checker.check_model(g)      # valida aqui (barato), não no Jetson (caro)
entradas = [(i.name, [d.dim_value or d.dim_param
                      for d in i.type.tensor_type.shape.dim]) for i in g.graph.input]
print('ONNX válido | opset', g.opset_import[0].version)
print('  entradas:', entradas)
print('  saídas  :', [o.name for o in g.graph.output])

DST = f'/content/drive/MyDrive/soja_rfdetr_{VARIANTE}_{SIZE}_base12k.onnx'
shutil.copy(onnx_path, DST)
print(f'\nONNX no Drive -> {DST} ({os.path.getsize(DST)/1e6:.0f} MB)')

print(f"""
=================== NO JETSON ===================
/usr/src/tensorrt/bin/trtexec \\
    --onnx=soja_rfdetr_{VARIANTE}_{SIZE}_base12k.onnx \\
    --saveEngine=soja_{VARIANTE}_{SIZE}_fp16.engine --fp16

python3 vigil_jetson.py --engine soja_{VARIANTE}_{SIZE}_fp16.engine \\
    --camera csi --roi {SIZE} --tiles 2 --esteira --laudo laudo.json

Confira o mapeamento de classe ANTES de confiar no resultado
(o off-by-one já apareceu 3 vezes neste projeto):
python3 vigil_jetson.py --engine ... --source video.mp4 --diag 60
=================================================""")

## 7. O que vem depois

Este modelo é a **linha de base**, e o valor dele depende inteiramente de a
câmara reproduzir o domínio do dataset. A ordem:

1. **Montar e calibrar o rig** (`jetson/calibrar_rig.py`). Se a régua devolver
   um px/mm diferente de 12,7, volte à célula de config, corrija `PX_POR_MM` e
   **reconstrua o dataset** — as cenas saem na escala errada em silêncio.
2. **Gerar o engine no Jetson** e medir fps a 704 px. Os ~28 fps são
   estimativa; a medição de 53,2 qps foi feita a 512 px.
3. **Rodar o modelo na câmara com soja de verdade.** É aqui que se descobre se
   a aposta funcionou: se o modelo base já inspeciona bem, o domínio foi mesmo
   reproduzido. Se não, a diferença que sobrou é o que o dataset próprio
   precisa cobrir — e agora dá para saber *o quê*, em vez de chutar.
4. **Capturar o dataset próprio** no rig, com bandeja de classe única (rótulo
   de graça via Otsu + pasta = classe) e bandeja mista anotada à mão para
   validação.
5. **Fine-tune a partir daqui** — este `.pth` vira o ponto de partida, não o
   COCO. E aí a comparação é limpa: base 12,5k sozinho × base + dataset
   próprio, no mesmo conjunto anotado.

**O que este notebook NÃO prova:** que a máquina tem X% de acurácia. Todo
número da §5 vem das mesmas 12,5k imagens do treino. A acurácia do produto só
existe depois do passo 4.